# Feline Skin Disease Detection - Model Statistics & Graphs

Colab version of `src/get_model_statistics_and_graphs.py`.

Aggregates per-class metrics, calibration error, and confusion-matrix heatmaps
across seeds for each architecture/approach.

Run all cells in order. A **GPU runtime** is recommended:
**Runtime → Change runtime type → GPU**.

To keep memory usage flat, the Keras session is cleared and garbage is
collected after every model — this is the main difference from the script,
and what lets it run in a memory-constrained environment.

## 1. Verify environment

In [ ]:
!pip install tensorflow-hub ml_insights

import tensorflow as tf
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {len(gpus)}")
for gpu in gpus:
    print(" ", gpu)

## 2. Mount Google Drive & clone repo

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Branch that contains the calibration + temperature code. Change if needed.
BRANCH = "update/temperature"

# Ensure we're in a valid directory before cleanup
os.chdir('/content')
!rm -rf /content/repo

# The repo tracks the .keras model files via Git LFS, but the LFS budget is exhausted,
# which makes the checkout abort partway and leave src/ missing (-> ModuleNotFoundError).
# We don't need the LFS blobs here (models are linked from Drive below), so tell git to
# skip the smudge filter and write lightweight pointer files instead. Setting this via
# os.environ (rather than an inline shell prefix) guarantees it reaches the git subprocess.
os.environ['GIT_LFS_SKIP_SMUDGE'] = '1'
!git clone -b {BRANCH} https://github.com/pelta-ai/feline-skin-disease-detection.git /content/repo
%cd /content/repo

# Fail loudly if the checkout still didn't produce the source tree.
assert os.path.isdir('/content/repo/src'), (
    "Checkout incomplete: /content/repo/src is missing. "
    "Check the clone output above for a git-lfs / checkout error."
)
print("OK: /content/repo/src present")

# Link Drive's final_data (train/val/test with images) into the repo
!rm -rf /content/repo/final_data
!ln -s /content/drive/MyDrive/feline-skin-disease-detection/final_data /content/repo/final_data

# constants.DATA_PATH defaults to "new_data", and count_image_classes.py reads
# new_data/train at *import* time (a module-level side effect). Those images aren't
# present in Colab, so point new_data at the same final_data tree to satisfy the import.
# The classifiers below are driven by DATA_DIR="final_data" regardless of this link.
!rm -rf /content/repo/new_data
!ln -s /content/drive/MyDrive/feline-skin-disease-detection/final_data /content/repo/new_data

In [ ]:
# Link the trained models stored on Drive into the repo. constants.TRAINED_MODELS_PATH
# resolves to ./trained_models relative to the repo, so the script logic finds them.
!rm -rf /content/repo/trained_models
!ln -sf /content/drive/MyDrive/feline-skin-disease-detection/trained_models /content/repo/trained_models
!ls /content/repo/trained_models/ | head

## 3. Compute statistics & graphs

In [ ]:
import os
import sys
import importlib

sys.path.insert(0, '/content/repo')
# If an earlier run imported (or tried to import) from /content/repo before the repo
# was fully checked out, Python cached that folder's listing and won't see src/ now.
# Invalidate the import caches so a freshly cloned src/ is picked up without a restart.
importlib.invalidate_caches()

import gc
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ml_insights as mli
import tensorflow as tf

from src.classifiers import ClassifierFactory
from src.classifiers.base_classifier import BaseClassifier
from src.utils import constants

# Config
architectures = ["convnext_tiny", "efficientnet_b0", "efficientnet_v2_b0", "mobilenet_v2", "mobilenet_v3_small", "nasnet_mobile", "resnet50"]
approaches = ["frozen", "finetuned"]
seeds = range(1, 16)  # Seeds 1 through 15

# Dataset directory linked from Drive above. The repo default (constants.DATA_PATH)
# is "new_data", which is NOT present in Colab, so we point the classifiers at the
# linked "final_data" folder (train/val/test with images) instead.
DATA_DIR = "final_data"

In [ ]:
for arch in architectures:
    for approach in approaches:
        all_cms, all_ps, all_rs, all_fs, all_eces_before, all_eces_after, all_y_true, all_y_prob, all_y_prob_cal, all_T = [], [], [], [], [], [], [], [], [], []

        print(f"\n--- Processing: {arch} ({approach}) ---")

        cnn = ClassifierFactory.create(arch, data_path=DATA_DIR)
        cnn.make_sub_datasets()

        for seed in seeds:
            # Matches format: resnet50_finetuned_seed2.keras
            # Note: strip underscores from arch name if your filenames don't use them (e.g., resnet50)
            clean_arch = arch.replace("_", "")
            filename = f"{clean_arch}_{approach}_seed_{seed}.keras"
            model_path = os.path.join(constants.TRAINED_MODELS_PATH, filename)

            if not os.path.exists(model_path):
                continue

            result = cnn.calibrate_and_evaluate(model_path=model_path, show_plots=False)

            # Collect data
            all_cms.append(result['confusion_matrix'])
            all_ps.append(result['per_class_precision'])
            all_rs.append(result['per_class_recall'])
            all_fs.append(result['per_class_f1'])
            all_eces_before.append(result['expected_calibration_error_before_calibration'])
            all_eces_after.append(result['expected_calibration_error_after_calibration'])
            all_y_true.append(result['y_true'])
            all_y_prob.append(result['y_prob'])
            all_y_prob_cal.append(result['y_prob_cal'])
            all_T.append(result['temperature'])

            # Free the loaded model before loading the next one to keep memory flat
            del result
            tf.keras.backend.clear_session()
            gc.collect()

        if not all_cms:
            del cnn
            tf.keras.backend.clear_session()
            gc.collect()
            continue

        # 1. Aggregate
        mean_cm = np.mean(all_cms, axis=0)
        avg_p, avg_r, avg_f1, avg_ece_before, avg_ece_after, avg_T = np.mean(all_ps, axis=0), np.mean(all_rs, axis=0), np.mean(all_fs, axis=0), np.mean(all_eces_before), np.mean(all_eces_after), np.mean(all_T)
        yt = np.concatenate(all_y_true)
        yp = np.concatenate(all_y_prob)
        ypc = np.concatenate(all_y_prob_cal)
        correct = (yp.argmax(1) == BaseClassifier._to_int(yt)).astype(int)

        # 2. Metrics Table
        summary_df = pd.DataFrame({
            'Class': cnn.class_names,
            'Precision': avg_p, 'Recall': avg_r, 'F1-Score': avg_f1,
            'Expected Calibration Error Before Calibration': avg_ece_before,
            'Expected Calibration Error After Calibration': avg_ece_after,
            'Temperature': avg_T,
        })
        print(summary_df.to_string(index=False))

        # 3. Display Graphs
        cm_title = (f'Overall CM: {arch} - {approach}\n(Mean of {len(all_cms)} seeds)')
        BaseClassifier.display_confusion_matrix(cm=mean_cm, class_names=cnn.class_names, title=cm_title)

        avg_rd_before_calib = mli.plot_reliability_diagram(correct, yp.max(1), show_histogram=True)
        rd_before_calib_title = (f'Overall RD Before Calibration: {arch} - {approach}\n(Pool of {len(all_eces_before)} seeds)')
        BaseClassifier.display_reliability_diagram(avg_rd_before_calib, rd_before_calib_title)

        avg_rd_after_calib = mli.plot_reliability_diagram(correct, ypc.max(1), show_histogram=True)
        rd_after_calib_title = (f'Overall RD After Calibration: {arch} - {approach}\n(Pool of {len(all_eces_after)} seeds)')
        BaseClassifier.display_reliability_diagram(avg_rd_after_calib, rd_after_calib_title)

        # Release this architecture before moving to the next
        del cnn
        tf.keras.backend.clear_session()
        gc.collect()